In [ ]:
import os, glob, json, math, shutil
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

import torch
from torch.utils.data import Dataset, ConcatDataset
import albumentations as A

# -----------------------------------------------------------------------------
# CONFIG
# -----------------------------------------------------------------------------
TRAIN_FOLDER = "patches_25m_balanced/train"
VAL_FOLDER   = "patches_25m_balanced/val"
HARD_NEG_FOLDER = "patches_25m/hard_negatives"

IGNITION_MODEL_PATH = "deep_unet_focal_25m.h5"  # <- CHANGE to your ignition-only model
LABEL_KEY = "class"
BATCH_SIZE = 8
PATCH_H, PATCH_W = 64, 64   # adjust if you use 192x192 etc.

# -----------------------------------------------------------------------------
# 1. U-NET MODEL (same as before)
# -----------------------------------------------------------------------------
def unet_deep(input_shape):
    inputs = layers.Input(shape=input_shape)

    # Encoder
    c1 = layers.Conv2D(32, 3, padding='same', activation='relu')(inputs)
    c1 = layers.Conv2D(32, 3, padding='same', activation='relu')(c1)
    p1 = layers.MaxPooling2D()(c1)

    c2 = layers.Conv2D(64, 3, padding='same', activation='relu')(p1)
    c2 = layers.Conv2D(64, 3, padding='same', activation='relu')(c2)
    p2 = layers.MaxPooling2D()(c2)

    c3 = layers.Conv2D(128, 3, padding='same', activation='relu')(p2)
    c3 = layers.Conv2D(128, 3, padding='same', activation='relu')(c3)
    p3 = layers.MaxPooling2D()(c3)

    # Bottleneck
    b  = layers.Conv2D(256, 3, padding='same', activation='relu')(p3)
    b  = layers.Conv2D(256, 3, padding='same', activation='relu')(b)

    # Decoder
    u3 = layers.UpSampling2D()(b)
    u3 = layers.Concatenate()([u3, c3])
    c4 = layers.Conv2D(128, 3, padding='same', activation='relu')(u3)
    c4 = layers.Conv2D(128, 3, padding='same', activation='relu')(c4)

    u2 = layers.UpSampling2D()(c4)
    u2 = layers.Concatenate()([u2, c2])
    c5 = layers.Conv2D(64, 3, padding='same', activation='relu')(u2)
    c5 = layers.Conv2D(64, 3, padding='same', activation='relu')(c5)

    u1 = layers.UpSampling2D()(c5)
    u1 = layers.Concatenate()([u1, c1])
    c6 = layers.Conv2D(32, 3, padding='same', activation='relu')(u1)
    c6 = layers.Conv2D(32, 3, padding='same', activation='relu')(c6)

    out = layers.Conv2D(1, 1, activation='sigmoid')(c6)
    return models.Model(inputs, out)


# -----------------------------------------------------------------------------
# 2. NPZ DATASET (same style as yours)
# -----------------------------------------------------------------------------
class NPZPatchDataset(Dataset):
    def __init__(self, folder, label_key="class", transforms=None, channel_stats=None):
        self.folder = folder
        self.files = sorted(glob.glob(os.path.join(folder, "patch_*.npz")))
        if len(self.files) == 0:
            raise FileNotFoundError(f"No patches found in {folder}")

        self.label_key = label_key
        self.transforms = transforms
        self.channel_stats = channel_stats
        self.skipped = []
        self.valid_files = []

        for f in self.files:
            try:
                arrs = np.load(f)
                if self.label_key in arrs.files:
                    self.valid_files.append(f)
                else:
                    self.skipped.append(os.path.basename(f))
            except Exception:
                self.skipped.append(os.path.basename(f))

        print(f"[{folder}] valid={len(self.valid_files)}, skipped={len(self.skipped)}")

        arrs = np.load(self.valid_files[0])
        feature_keys = [k for k in arrs.files if k != self.label_key]
        self.C = len(feature_keys)

    def __len__(self):
        return len(self.valid_files)

    def __getitem__(self, idx):
        f = self.valid_files[idx]
        arrs = np.load(f)
        feature_keys = [k for k in arrs.files if k != self.label_key]

        X = np.stack([arrs[k] for k in feature_keys], axis=-1).astype(np.float32)
        # ---- DROP LAST TWO CHANNELS ----
        # if X.shape[-1] > 2:
        #     X = X[..., :-2]     # keep all except last 2
        # else:
        #     raise ValueError("Dataset has fewer than 2 channels — cannot drop last two.")
        y = arrs[self.label_key].astype(np.float32)

        # Normalize
        if self.channel_stats is not None:
            mean = self.channel_stats["mean"][:X.shape[-1]]
            std  = self.channel_stats["std"][:X.shape[-1]]
            X = (X - mean) / (std + 1e-6)

        # Albumentations
        if self.transforms:
            out = self.transforms(image=X, mask=y)
            X, y = out["image"], out["mask"]

        X = torch.from_numpy(X.transpose(2, 0, 1))  # (C,H,W)
        y = torch.from_numpy(y[None, ...])          # (1,H,W)
        return X, y


# -----------------------------------------------------------------------------
# 3. CHANNEL STATS WITH STD CLAMPING
# -----------------------------------------------------------------------------
def compute_channel_stats(folder, label_key="class", max_files=300):
    files = sorted(glob.glob(os.path.join(folder, "patch_*.npz")))
    files = files[:min(max_files, len(files))]

    mean_acc, M2_acc, count = None, None, 0

    for i, f in enumerate(files, 1):
        arrs = np.load(f)
        feature_keys = [k for k in arrs.files if k != label_key]
        X = np.stack([arrs[k] for k in feature_keys], axis=-1).astype(np.float64)

        N = X.shape[0] * X.shape[1]
        x_mean = X.reshape(-1, X.shape[-1]).mean(0)
        x_var  = X.reshape(-1, X.shape[-1]).var(0)

        if mean_acc is None:
            mean_acc = x_mean
            M2_acc   = x_var * N
            count    = N
        else:
            delta = x_mean - mean_acc
            tot = count + N
            mean_acc = mean_acc + delta * N / tot
            M2_acc   = M2_acc + x_var * N + delta**2 * (count * N / tot)
            count = tot

    std = np.sqrt(M2_acc / count + 1e-6)
    return {"mean": mean_acc, "std": std}

print("Computing channel stats on train patches…")
stats_raw = compute_channel_stats(TRAIN_FOLDER, label_key=LABEL_KEY)
mean = np.array(stats_raw["mean"], dtype=np.float32)
std  = np.array(stats_raw["std"],  dtype=np.float32)

# Clamp tiny std to avoid huge scaling
std_floor = 0.1
std = np.where(std < std_floor, std_floor, std)

stats = {"mean": mean, "std": std}
with open("channel_stats.json", "w") as f:
    json.dump({k: v.tolist() for k, v in stats.items()}, f, indent=2)
print("Channel stats saved.")

In [ ]:
# -----------------------------------------------------------------------------
# 4. DATASETS + TRANSFORMS
# -----------------------------------------------------------------------------
train_tfms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
])

val_tfms = None

train_ds = NPZPatchDataset(TRAIN_FOLDER, label_key=LABEL_KEY,
                           transforms=train_tfms, channel_stats=stats)
val_ds   = NPZPatchDataset(VAL_FOLDER,   label_key=LABEL_KEY,
                           transforms=val_tfms,   channel_stats=stats)

C = train_ds.C
H, W = PATCH_H, PATCH_W

print("Train patches:", len(train_ds))
print("Val patches:", len(val_ds))

In [ ]:
# -----------------------------------------------------------------------------
# 5. KERAS GENERATOR (NaN-safe)
# -----------------------------------------------------------------------------
def keras_generator(dataset, batch_size=8, shuffle=True):
    n = len(dataset)
    indices = np.arange(n)

    while True:
        if shuffle:
            np.random.shuffle(indices)

        for start in range(0, n, batch_size):
            batch_idx = indices[start:start+batch_size]

            X_list, y_list = [], []
            for i in batch_idx:
                x_t, y_t = dataset[i]
                x_np = x_t.numpy().transpose(1, 2, 0)
                y_np = y_t.numpy().transpose(1, 2, 0)

                # Clean any nan/inf just in case
                x_np = np.nan_to_num(x_np, nan=0.0, posinf=0.0, neginf=0.0)
                y_np = np.nan_to_num(y_np, nan=0.0, posinf=0.0, neginf=0.0)

                X_list.append(x_np)
                y_list.append(y_np)

            if not X_list:
                continue

            X = np.stack(X_list).astype(np.float32)
            y = np.stack(y_list).astype(np.float32)
            yield X, y

In [ ]:
# -----------------------------------------------------------------------------
# 6. LOSSES: FOCAL + PATCH-LEVEL SUPPRESSION (STAGED)
# -----------------------------------------------------------------------------
def focal_loss(alpha=0.75, gamma=1.5):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-6, 1 - 1e-6)
        ce = - (alpha * y_true * tf.math.log(y_pred) +
                (1 - alpha) * (1 - y_true) * tf.math.log(1 - y_pred))
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        fl = (1 - pt) ** gamma * ce
        return tf.reduce_mean(fl)
    return loss

def patch_level_fp_loss(y_true, y_pred):
    """
    BCE on patch-level "any activation" vs "any ground truth".
    """
    y_pred_any = tf.reduce_max(y_pred, axis=[1, 2, 3])
    y_true_any = tf.reduce_max(y_true, axis=[1, 2, 3])

    eps = 1e-6
    y_pred_any = tf.clip_by_value(y_pred_any, eps, 1.0 - eps)

    bce = - (y_true_any * tf.math.log(y_pred_any) +
             (1 - y_true_any) * tf.math.log(1 - y_pred_any))
    return tf.reduce_mean(bce)

def combined_loss(alpha=0.75, gamma=1.5, lambda_patch=0.5):
    f = focal_loss(alpha=alpha, gamma=gamma)
    def loss(y_true, y_pred):
        return f(y_true, y_pred) + lambda_patch * patch_level_fp_loss(y_true, y_pred)
    return loss


# -----------------------------------------------------------------------------
# 7. METRICS (pixel & patch FP)
# -----------------------------------------------------------------------------
def pixel_fp_rate(threshold=0.5):
    def metric(y_true, y_pred):
        y_true_f = tf.reshape(y_true, [-1])
        y_pred_f = tf.reshape(y_pred, [-1])
        y_pred_bin = tf.cast(y_pred_f > threshold, tf.float32)
        neg_mask = 1.0 - y_true_f
        fp = tf.reduce_sum(y_pred_bin * neg_mask)
        neg = tf.reduce_sum(neg_mask)
        return tf.math.divide_no_nan(fp, neg)
    metric.__name__ = f"pixel_fp@{threshold}"
    return metric

def patch_fp_rate(threshold=0.5):
    def metric(y_true, y_pred):
        y_pred_bin = tf.cast(y_pred > threshold, tf.float32)
        y_true_any = tf.reduce_max(y_true, axis=[1, 2, 3])
        y_pred_any = tf.reduce_max(y_pred_bin, axis=[1, 2, 3])
        neg_mask = 1.0 - y_true_any
        fp_patches = tf.reduce_sum(neg_mask * y_pred_any)
        num_neg = tf.reduce_sum(neg_mask)
        return tf.math.divide_no_nan(fp_patches, num_neg)
    metric.__name__ = f"patch_fp@{threshold}"
    return metric


In [ ]:
# -----------------------------------------------------------------------------
# 9. LOAD IGNITION-ONLY MODEL AND RUN 4-PHASE TRAINING
# -----------------------------------------------------------------------------
# Generators
train_gen = keras_generator(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_gen   = keras_generator(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

steps_per_epoch  = math.ceil(len(train_ds) / BATCH_SIZE)
val_steps        = math.ceil(len(val_ds) / BATCH_SIZE)

# Load ignition-only model (trained before on ignition patches with focal)
print(f"Loading ignition-only model from {IGNITION_MODEL_PATH}")
model = tf.keras.models.load_model("deep_unet_focal_25m.h5", compile=False)
print("Model loaded.")

opt = tf.keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0)

In [ ]:
# ------------------ PHASE 1: FOCAL ONLY ON MIXED DATA ------------------------
print("\n🚀 Phase 1: focal only on (ignition + non-ignition) patches")
model.compile(
    optimizer=opt,
    loss=focal_loss(alpha=0.9, gamma=2.5),
    metrics=[
        'precision',
        'recall',
        pixel_fp_rate(0.5),
        patch_fp_rate(0.5),
    ]
)

history_p1 = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    validation_steps=val_steps,
    epochs=3,   # 3–5 epochs
    verbose=1
)

In [ ]:
model.save("deep_unet_focal_25m_p2.h5")
print("Model saved: deep_unet_focal_25m_p2.h5")

In [ ]:
model = tf.keras.models.load_model("deep_unet_focal_25m_p2.h5", compile=False)

In [ ]:
print("\n🔥 Phase 2: focal + weak patch suppression (λ=0.1)")
model.compile(
    optimizer=opt,
    loss=combined_loss(alpha=0.9, gamma=2.5, lambda_patch=0.1),
    metrics=[
        'precision',
        'recall',
        pixel_fp_rate(0.5),
        patch_fp_rate(0.5),
    ]
)

history_p2 = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    validation_steps=val_steps,
    epochs=5,   # 3–5
    verbose=1
)

In [ ]:
# ---------------- PHASE 3: FOCAL + STRONGER SUPPRESSION ----------------------
print("\n🔥 Phase 3: focal + stronger suppression (λ=0.5)")
model.compile(
    optimizer=opt,
    loss=combined_loss(alpha=0.9, gamma=2.5, lambda_patch=0.5),
    metrics=[
        'precision',
        'recall',
        pixel_fp_rate(0.5),
        patch_fp_rate(0.5),
    ]
)

history_p3 = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    validation_steps=val_steps,
    epochs=5,
    verbose=1
)

In [ ]:
model.save("deep_unet_focal_25m_p3.h5")
print("Model saved: deep_unet_focal_25m_p3.h5")

In [ ]:
import os, glob, shutil
import numpy as np

# -------------------- Recall-friendly HNM -------------------- #
def collect_hard_negatives_recall_friendly(
    model,
    base_dataset,
    stats,
    out_folder,
    label_key="class",
    top_k=300,
    min_score=0.3,   # only mine FPs with max_pred >= this
):
    """
    Recall-optimized HNM:
    - Only mine the *worst* false positives (most confident FPs).
    - Keep K small (e.g., 300) to avoid over-suppressing recall.
    """
    os.makedirs(out_folder, exist_ok=True)
    scores = []

    for f in base_dataset.valid_files:
        arrs = np.load(f)
        y = arrs[label_key].astype(np.float32)

        # only look at non-ignition patches
        if y.max() > 0:
            continue

        feature_keys = [k for k in arrs.files if k != label_key]
        X = np.stack([arrs[k] for k in feature_keys], axis=-1).astype(np.float32)
        # if X.shape[-1] > 2:
        #     X = X[..., :-2]     # keep all except last 2
        # else:
        #     raise ValueError("Dataset has fewer than 2 channels — cannot drop last two.")
        # normalize
        mean = stats["mean"][:X.shape[-1]]
        std  = stats["std"][:X.shape[-1]]
        X = (X - mean) / (std + 1e-6)
        Xb = X[np.newaxis, ...]

        pred = model.predict(Xb, verbose=0)[0]
        max_pred = float(pred.max())
        scores.append((f, max_pred))

    if not scores:
        print("No negative patches to mine from.")
        return

    # sort descending by max_pred (most overconfident negatives first)
    scores.sort(key=lambda x: x[1], reverse=True)

    # keep only those above min_score
    if min_score is not None:
        scores = [s for s in scores if s[1] >= min_score]

    if not scores:
        print(f"No negatives with score >= {min_score}, skipping HNM.")
        return

    top_k = min(top_k, len(scores))
    selected = scores[:top_k]

    print(f"Selected {len(selected)} hard negatives (top_k={top_k}, min_score={min_score}).")

    # copy them into folder
    for f, s in selected:
        dst = os.path.join(out_folder, os.path.basename(f))
        shutil.copy2(f, dst)

    print(f"Hard negatives copied to: {out_folder}")


In [ ]:
# -------------------- Phase 4: recall-optimized training -------------------- #

print("\n🔎 Phase 4 (recall-optimized): soft hard-negative mining")

# 1) Mine hard negatives in a recall-friendly way
# collect_hard_negatives_recall_friendly(
#     model,
#     train_ds,
#     stats,
#     out_folder="patches_25m_balanced/hard_negatives",
#     label_key=LABEL_KEY,
#     top_k=300,      # smaller: don't crush recall
#     min_score=0.3   # only really confident FPs
# )

hard_neg_files = glob.glob(os.path.join(HARD_NEG_FOLDER, "patch_*.npz"))
if hard_neg_files:
    hard_ds = NPZPatchDataset(
        HARD_NEG_FOLDER,
        label_key=LABEL_KEY,
        transforms=train_tfms,   # same augmentations
        channel_stats=stats
    )

    # Boost hard negatives moderately (2x), not aggressively
    from torch.utils.data import ConcatDataset
    recall_train_ds = ConcatDataset([
        train_ds,
        hard_ds,
        hard_ds,   # 2x weight for hard negatives
    ])

    print("Recall-optimized combined train size:", len(recall_train_ds))

    train_gen_recall = keras_generator(recall_train_ds, batch_size=BATCH_SIZE, shuffle=True)
    steps_per_epoch_recall = math.ceil(len(recall_train_ds) / BATCH_SIZE)

    # 2) Compile with recall-friendly loss
    print("\n🔥 Phase 4 training: focal(α=0.85, γ=2.0) + 0.3 * patch_loss")

    opt = tf.keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0)

    model.compile(
        optimizer=opt,
        loss=combined_loss(alpha=0.9, gamma=2.5, lambda_patch=0.3),
        metrics=[
            'precision',
            'recall',
            pixel_fp_rate(0.5),
            patch_fp_rate(0.5),
        ]
    )

    history_p4 = model.fit(
        train_gen_recall,
        steps_per_epoch=steps_per_epoch_recall,
        validation_data=val_gen,
        validation_steps=val_steps,
        epochs=3,        # keep short to preserve recall
        verbose=1
    )
else:
    print("No hard negatives found for Phase 4; skipping recall-optimized training.")


In [ ]:
model.save("deep_unet_focal_25m_p4.h5")
print("Model saved: deep_unet_focal_25m_p4.h5")

In [ ]:
def evaluate_thresholds_recall_first(model, val_ds, stats, thresholds):
    """
    For each threshold:
      - compute ignition recall (patch-wise)
      - compute patch_fp_rate
      - compute score = recall - 0.5 * patch_fp_rate
    """
    results = []

    for t in thresholds:
        ign_total = 0
        ign_detected = 0
        neg_total = 0
        neg_fp = 0

        for f in val_ds.valid_files:
            arr = np.load(f)
            y = arr[LABEL_KEY].astype(np.float32)
            feature_keys = [k for k in arr.files if k != LABEL_KEY]
            X = np.stack([arr[k] for k in feature_keys], axis=-1).astype(np.float32)
            # if X.shape[-1] > 2:
            #     X = X[..., :-2]     # keep all except last 2
            # else:
            #     raise ValueError("Dataset has fewer than 2 channels — cannot drop last two.")
            mean = stats["mean"][:X.shape[-1]]
            std  = stats["std"][:X.shape[-1]]
            X = (X - mean) / (std + 1e-6)

            pred = model.predict(X[np.newaxis, ...], verbose=0)[0]
            pred_bin = (pred >= t).astype(np.float32)

            if y.max() > 0:  # ignition patch
                ign_total += 1
                ign_detected += int(pred_bin.max() == 1)
            else:            # non-ignition patch
                neg_total += 1
                neg_fp += int(pred_bin.max() == 1)

        if ign_total == 0 or neg_total == 0:
            continue

        recall = ign_detected / ign_total
        patch_fp_rate = neg_fp / neg_total
        score = recall - 0.5 * patch_fp_rate   # recall-first score

        results.append({
            "threshold": float(t),
            "ignition_recall": recall,
            "patch_fp_rate": patch_fp_rate,
            "score": score
        })

    return results


In [ ]:
thresholds = np.linspace(0.2, 0.9, 15)
results = evaluate_thresholds_recall_first(model, val_ds, stats, thresholds)

for r in results:
    print(
        f"t={r['threshold']:.2f} | "
        f"recall={r['ignition_recall']:.3f} | "
        f"patch_fp={r['patch_fp_rate']:.3f} | "
        f"score={r['score']:.3f}"
    )

best = max(results, key=lambda r: r["score"])
print("\n🏆 Best threshold (recall-first):")
print(best)


In [ ]:
model.save("deep_unet_focal_25m_p5.h5")
print("Model saved: deep_unet_focal_25m_p5.h5")

In [ ]:
model = tf.keras.models.load_model("deep_unet_focal_25m_p4.h5", compile=False)